[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-04-automodel-autotokenizer.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · AutoModel and AutoTokenizer — Loading Any Model from the Hub
**certified-journeys / huggingface-nlp-certified** · Day 4 · Model Loading

> **Goal for today:** By the end of this notebook you can load any model and tokenizer from the Hugging Face Hub using Auto Classes, run a manual forward pass, and inspect model outputs including logits and hidden states.


In [ ]:
%pip install -q transformers torch


## Step 1 · What are Auto Classes?

The Hugging Face `transformers` library ships hundreds of model architectures. Auto Classes let you load **any** of them with a single API — no need to know in advance whether you're loading BERT, RoBERTa, DistilBERT, or GPT-2.

| Class | Purpose |
|---|---|
| `AutoTokenizer` | Tokenizer matched to any model checkpoint |
| `AutoModel` | Base model (no task head) |
| `AutoModelForSequenceClassification` | Base + classification head |
| `AutoModelForTokenClassification` | Base + per-token head (NER) |
| `AutoModelForQuestionAnswering` | Base + span-extraction head |

Each `.from_pretrained(checkpoint)` call:
1. Downloads the model config from the Hub
2. Reads `"model_type"` from `config.json`
3. Instantiates the matching architecture and loads weights

**Reference:** [Auto Classes documentation](https://huggingface.co/docs/transformers/model_doc/auto)


In [ ]:
from transformers import AutoTokenizer, AutoModel

checkpoint = "distilbert-base-uncased"

# Download tokenizer — reads tokenizer_config.json on the Hub
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Download model weights — ~66M parameters for DistilBERT
model = AutoModel.from_pretrained(checkpoint)

print("Tokenizer class:", type(tokenizer).__name__)
print("Model class    :", type(model).__name__)
print("Hidden size    :", model.config.hidden_size)
print("Num layers     :", model.config.num_hidden_layers)


### What just happened?
- **`AutoTokenizer.from_pretrained`** inspected the Hub checkpoint and returned a `DistilBertTokenizerFast` — the fast (Rust-backed) variant.
- **`AutoModel.from_pretrained`** returned a `DistilBertModel` — the bare transformer with no task head.
- `model.config` exposes every architectural hyperparameter; `hidden_size=768` means each token becomes a 768-dim vector.
- **Model weights are cached** in `~/.cache/huggingface/hub` after the first download.


## Step 2 · Tokenizing text and understanding the output

Before we can run a forward pass, text must be converted into token IDs. The tokenizer:
1. **Splits** text into subword tokens (`WordPiece` for BERT family)
2. **Maps** tokens to integer IDs using a vocabulary
3. **Adds special tokens** — `[CLS]` at position 0, `[SEP]` at the end
4. Returns PyTorch/TF tensors (or lists) via `return_tensors`

| Key output | Meaning |
|---|---|
| `input_ids` | Integer ID for each token |
| `attention_mask` | 1 = real token, 0 = padding |
| `token_type_ids` | Segment A vs B (only some models) |


In [ ]:
sentence = "Hugging Face makes NLP accessible to every engineer."

# return_tensors='pt' → PyTorch tensors; use 'tf' for TensorFlow
inputs = tokenizer(sentence, return_tensors="pt")

print("Keys returned by tokenizer:", list(inputs.keys()))
print("input_ids shape:", inputs["input_ids"].shape)  # (batch=1, seq_len)

# Decode IDs back to tokens to see what happened
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)
print("IDs:   ", inputs["input_ids"][0].tolist())


### What just happened?
- **`[CLS]`** (ID 101) is always prepended. For classification tasks its embedding is the aggregate representation of the whole sentence.
- **`[SEP]`** (ID 102) marks the end of each sequence.
- Subword tokenization means `"accessible"` may split into `["accessible"]` (common word) but rare words like `"NLP"` become `["nl", "##p"]` — the `##` prefix means "continuation of previous token".
- `attention_mask` is all 1s here (single short sentence, no padding).


## Step 3 · Running a forward pass and extracting the [CLS] embedding

The base `AutoModel` (no task head) returns:
- **`last_hidden_state`** — shape `(batch, seq_len, hidden_size)` — one vector per token
- **`pooler_output`** (BERT only) — the `[CLS]` vector passed through a linear + tanh

For sentence-level tasks you usually extract **position 0** of `last_hidden_state` (the `[CLS]` token) or use `pooler_output`.

**Reference:** [Quick tour — forward pass](https://huggingface.co/docs/transformers/en/quicktour)


In [ ]:
import torch

# torch.no_grad() disables gradient computation — saves memory & speeds up inference
with torch.no_grad():
    # **inputs unpacks the dict into keyword arguments: input_ids=..., attention_mask=...
    outputs = model(**inputs)

print("Output keys:", list(outputs.keys()))
print("last_hidden_state shape:", outputs.last_hidden_state.shape)
# Shape: (batch_size=1, seq_len, hidden_size=768)

# Extract the [CLS] token embedding (position 0)
cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, 768)
print("[CLS] embedding shape:", cls_embedding.shape)
print("First 5 values:", cls_embedding[0, :5].tolist())


### What just happened?
- **`torch.no_grad()`** tells PyTorch not to build the computation graph — this alone can halve memory usage and speed up inference by ~30%. Always use it when you're not training.
- `**inputs` unpacks the tokenizer dict as keyword arguments — this is the canonical pattern across all HF models.
- `last_hidden_state[:, 0, :]` slices: all batches, position 0 (`[CLS]`), all hidden dims — giving a 768-dimensional sentence embedding.
- **`[CLS]` is not magic** — it only carries sentence-level meaning after fine-tuning on a classification task (or with a sentence-transformer training objective).


## Step 4 · Loading a task-specific head with AutoModelForSequenceClassification

For downstream tasks you almost always want a **task-specific Auto class**, not the base `AutoModel`. These classes add a task head (e.g. a linear classifier) on top of the encoder.

| Auto class | What it adds on top of the encoder |
|---|---|
| `AutoModelForSequenceClassification` | Linear(`hidden_size → num_labels`) |
| `AutoModelForTokenClassification` | Linear per token |
| `AutoModelForMaskedLM` | Vocabulary projection head |

A **fine-tuned checkpoint** already has trained head weights — just pass it to `.from_pretrained()`.


In [ ]:
from transformers import AutoModelForSequenceClassification

# Fine-tuned DistilBERT checkpoint for SST-2 sentiment (positive / negative)
clf_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

clf_tokenizer = AutoTokenizer.from_pretrained(clf_checkpoint)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_checkpoint)

print("Model class:", type(clf_model).__name__)
# The model automatically exposes label names when fine-tuned
print("id2label   :", clf_model.config.id2label)
print("num_labels :", clf_model.config.num_labels)


### What just happened?
- `AutoModelForSequenceClassification` returns a `DistilBertForSequenceClassification` — the encoder plus a two-layer classifier head.
- `config.id2label` maps integer logit positions to human-readable label names, baked in during fine-tuning.
- **Always pick the task-specific Auto class** over the base `AutoModel` for downstream tasks — the base class has no classification head so you'd need to add and train one yourself.


## Step 5 · Manual forward pass: from text to predicted label

Now we put it all together: tokenize → forward pass under `no_grad` → read logits → argmax → label name.

**Logits** are the raw unnormalised scores before softmax. For classification:
- Shape: `(batch_size, num_labels)`
- `argmax(-1)` gives the predicted class index
- Wrap in softmax if you need calibrated probabilities


In [ ]:
import torch
import torch.nn.functional as F

sentences = [
    "This library is absolutely fantastic!",
    "I am deeply disappointed with this model.",
    "The results are neither good nor bad.",
]

# Batch tokenization: padding=True aligns all sequences to the same length
batch_inputs = clf_tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)
print("Batch input_ids shape:", batch_inputs["input_ids"].shape)  # (3, max_len)

with torch.no_grad():
    batch_outputs = clf_model(**batch_inputs)

logits = batch_outputs.logits
print("Logits shape:", logits.shape)  # (3, 2)  — 3 sentences, 2 labels

# Softmax → probabilities; dim=-1 operates over the label dimension
probs = F.softmax(logits, dim=-1)
predictions = logits.argmax(dim=-1)

for i, sentence in enumerate(sentences):
    label = clf_model.config.id2label[predictions[i].item()]
    confidence = probs[i][predictions[i]].item()
    print(f"  [{label:>8s} {confidence:.1%}] {sentence}")


### What just happened?
- **Batch tokenization** with `padding=True` pads shorter sequences so all inputs are the same length — `attention_mask` tells the model which positions are real vs padding.
- `logits` has shape `(batch=3, num_labels=2)` — one score per label per sentence.
- **`argmax(dim=-1)`** finds the highest-scoring label for each sentence (argmax over the last dimension).
- **`torch.no_grad()`** matters even more in batch mode: for a batch of 3 sequences at inference, skipping gradient tracking saves ~50% of the memory that would otherwise store the computation graph.


In [ ]:
# Challenge: Load any zero-shot classification model and run inference
# on three sentences of your choice. Print the label and confidence.
#
# Suggested checkpoint: "facebook/bart-large-mnli"
# Use AutoModelForSequenceClassification + AutoTokenizer
#
# Scaffold:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
#
# checkpoint = "facebook/bart-large-mnli"
# candidate_labels = ["technology", "sports", "politics"]
#
# tokenizer = ...
# model = ...
#
# sentence = "The new transformer architecture outperforms all previous benchmarks."
# # Hint: for zero-shot, the model expects pairs (sentence, hypothesis)
# # where hypothesis = f"This example is about {label}."
# # Loop over candidate_labels, run forward pass, compare entailment logits
#
# Your solution here


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `AutoTokenizer.from_pretrained` | Downloads and returns the right tokenizer class for any checkpoint |
| `AutoModel` vs task-specific | Base model has no head — always use task-specific Auto class for downstream tasks |
| `[CLS]` token | Position 0 in last_hidden_state; carries sentence-level meaning after fine-tuning |
| `last_hidden_state` | Shape (batch, seq_len, hidden_size) — one vector per input token |
| `logits` | Raw unnormalised scores; argmax → predicted class; softmax → probabilities |
| `torch.no_grad()` | Disables gradient computation at inference — saves memory and speeds up forward pass |
| `**inputs` dict unpacking | Standard pattern for passing tokenizer output to any HF model |

> **Tip:** AutoModelForSequenceClassification automatically adds a classification head on top of the base model — always pick the task-specific Auto class over the base AutoModel for downstream tasks.

---
## What's next
**Day 5** → The Datasets Library — Loading, Filtering, Mapping, and Tokenizing large corpora efficiently with `load_dataset`, `dataset.filter`, and `dataset.map`.

Mark Day 4 complete in your [tracker](../index.html).
